# Continuidad escolar en Bolivia
## Dos resultados educativos y cuatro perfiles municipales

**Caso 1 · Maestría en Inteligencia Artificial, Machine Learning y Data Science**

**Pregunta.** ¿Qué características ayudan a identificar diferencias de asistencia entre adolescentes de 12 a 17 años y dónde conviene investigar las barreras de continuidad?

**Lectura del indicador.** La EH2025 observa matrícula y asistencia en el momento de la encuesta. Para evaluar la clase de interés se usa inasistencia = 1 − Asistencia_Escolar. Confirmar deserción exige seguir la trayectoria de matrícula, traslado y egreso entre periodos.

Este cuaderno continúa el [análisis original en Colab](https://colab.research.google.com/github/anguloramiro-del/Proyecto_EH2025_ML_Caso1/blob/b2f31d362bcca1d727014c045a0309256cbb77a6/notebooks/03_Caso1_Completo.ipynb). Se puede leer sin conectar un entorno. Las celdas de visualización descargan resultados públicos del proyecto. La última sección permite reproducir el nuevo modelo dentro del entorno que contiene el parquet analítico privado.

## 1 · Asistencia observada y capacidad predictiva

En la EH2025, 169 de 4.731 adolescentes registraron inasistencia. Al aplicar los factores de expansión, la estimación nacional es **4,16 %** (IC 95 % aproximado ajustado por estratos con una sola UPM: 2,84–5,47 %). La tasa por edad pasa de 1,29 % a los 12–13 años a 4,08 % a los 14–15 y 7,39 % a los 16–17.

| Modelo | AUC-ROC en reserva | Precisión promedio (AP) | Sensibilidad al umbral 0,5 |
|---|---:|---:|---:|
| Random Forest | 0,703 | 0,084 | 0 % |
| XGBoost | 0,713 | 0,110 | 0 % |

XGBoost obtuvo la mayor AP en validación cruzada agrupada por UPM. En la reserva de 945 adolescentes, la AP de referencia fue 0,033. Ninguno de los dos modelos detectó los 34 casos positivos al umbral fijado. Los puntajes permiten estudiar asociaciones, pero su uso como alerta individual requiere otra validación.

**¿Qué factores destacan?** En XGBoost, departamento, fuente de agua y área urbana/rural aportaron más ganancia. En Random Forest destacaron ingreso por persona, departamento y edad. La importancia depende del modelo y no estima cuánto cambiaría la asistencia al intervenir sobre una variable.

## 2 · Nivel educativo alcanzado

Se conservaron las cinco categorías de `niv_ed` observadas a los 12–17 años. Se generó una predicción fuera de muestra por adolescente con cinco pliegues estratificados agrupados por UPM. La imputación y la codificación se ajustaron en cada entrenamiento. Se excluyeron preguntas educativas, asistencia e identificadores de los predictores.

| Medida ponderada | Random Forest | Predecir siempre la categoría mayoritaria del entrenamiento |
|---|---:|---:|
| Exactitud | 79,0 % | 73,2 % |
| F1 macro | 0,327 | 0,169 |

| Nivel observado | Casos muestrales | Sensibilidad ponderada |
|---|---:|---:|
| Ninguno | 17 | 0,0 % |
| Primaria incompleta | 482 | 31,2 % |
| Primaria completa | 722 | 27,9 % |
| Secundaria incompleta | 3.497 | 97,6 % |
| Secundaria completa | 13 | 0,0 % |

El modelo reconoce el nivel más frecuente. La sensibilidad baja en primaria y nula en las dos categorías menos numerosas limita la interpretación de la exactitud global. La escolaridad alcanzada a una edad concreta tampoco determina si una persona abandonó sus estudios.

## 3 · Mapa municipal y agrupamiento

El análisis territorial utiliza el **Censo 2024**, con una observación por municipio o TIOC. K-Means combina cuatro variables estandarizadas: no asistencia de 12–17 años, ruralidad de ese grupo, años de escolaridad de adultos de 19 años o más y carencia de fuente mejorada de agua entre adolescentes. Cada jurisdicción tiene el mismo peso en el ajuste.

| Perfil | Municipios/TIOC | Tasa de no asistencia agrupada por población |
|---|---:|---:|
| 1 | 82 | 4,35 % |
| 2 | 157 | 7,63 % |
| 3 | 41 | 8,65 % |
| 4 | 63 | 20,71 % |

Cuatro grupos dieron una silueta de 0,3189 frente a 0,3187 con cinco. La delimitación es exploratoria. La no asistencia participa en el agrupamiento y las diferencias entre perfiles forman parte de su construcción. En el perfil 4, **24.976 de 120.621** adolescentes registraron no asistencia. En el perfil 1 la tasa fue menor, pero el número absoluto fue mayor (**37.415**). Presupuestar por tasa y por volumen puede conducir a prioridades distintas.

Ejecuta la celda para ver el mapa, alternar entre tasa y perfiles y descargar una copia interactiva.

In [ ]:
from IPython.display import HTML, display
from pathlib import Path
from urllib.request import urlopen
from html import escape

base_publica = "https://raw.githubusercontent.com/anguloramiro-del/Proyecto_EH2025_ML_Caso1/b2f31d362bcca1d727014c045a0309256cbb77a6/"
ruta_mapa = "outputs/municipal/mapa_municipal_2024.html"
with urlopen(base_publica + ruta_mapa, timeout=90) as respuesta:
    mapa = respuesta.read().decode("utf-8")
Path("mapa_municipal_2024.html").write_text(mapa, encoding="utf-8")
display(HTML('<iframe title="Mapa municipal interactivo" width="100%" height="760" srcdoc="' +
             escape(mapa, quote=True) + '"></iframe>'))
# En Colab, el panel Archivos permite descargar mapa_municipal_2024.html.

## 4 · Tres acciones para contrastar en territorio

| Evidencia | Acción propuesta | Indicador de seguimiento |
|---|---|---|
| La inasistencia ponderada llega al 7,39 % a los 16–17 años. | Las direcciones distritales pueden contrastar matrícula y asistencia con las unidades educativas y consultar a las familias sobre barreras de continuidad, traslado o egreso. | Proporción con situación verificada y proporción que continúa o retorna a los 30 y 60 días. |
| El perfil 4 reúne 63 jurisdicciones y 20,71 % de no asistencia agrupada. El perfil 1 concentra 37.415 casos. | Seleccionar pilotos con dos criterios explícitos: intensidad del problema y cantidad de adolescentes. Contrastar los datos de 2024 con registros locales recientes. | Cobertura por perfil y número de adolescentes revinculados sobre los contactados. |
| Fuente de agua y ruralidad figuran entre los predictores destacados de XGBoost; el perfil 3 tiene en promedio 48,47 % sin fuente mejorada de agua. | En una selección de las 41 jurisdicciones del perfil 3, verificar con estudiantes si los servicios, los tiempos de traslado u otras barreras afectan la asistencia antes de financiar una intervención. | Barreras confirmadas, cobertura del apoyo elegido y cambio de asistencia frente a una línea de base. |

Fuente individual: Instituto Nacional de Estadística, [Encuesta de Hogares 2025](https://anda.ine.gob.bo/index.php/catalog/256), acceso documentado el 15/11/2026. Fuente municipal: tabulados del Censo 2024 del INE y cartografía referencial del Atlas UDAPE. Los años y universos difieren. El análisis no vincula personas de la encuesta con municipios censales.

## 5 · Reproducir el modelo de nivel educativo

La función usa el archivo `outputs/private/analysis.parquet` producido por el proyecto original. La entrada permanece en el entorno autorizado. Solo se presentan métricas agregadas. La clase mayoritaria se determina dentro de cada pliegue con los factores de expansión.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder

def evaluar_nivel(ruta):
    numericas = ['s01a_03', 'yhogpc', 'hh_totper']
    categoricas = ['s01a_02', 'depto', 'area', 'hh_s06a_07', 'hh_s06a_09', 'hh_s06a_10', 'hh_s06a_12']
    columnas = numericas + categoricas
    datos = pd.read_parquet(ruta, columns=columnas + ['niv_ed', 'factor', 'upm'])
    datos = datos.loc[datos.s01a_03.between(12, 17) & datos.niv_ed.notna()].copy()
    etiquetas = {0:'Ninguno', 1:'Primaria incompleta', 2:'Primaria completa',
                 3:'Secundaria incompleta', 4:'Secundaria completa', 5:'Superior', 6:'Otros'}
    assert datos.niv_ed.isin(etiquetas).all() and datos.factor.gt(0).all() and datos.upm.notna().all()
    y = datos.niv_ed.astype(int).to_numpy()
    w = datos.factor.to_numpy()
    clases = sorted(set(y))
    pred = np.full(len(y), -1)
    base = pred.copy()
    grupos = datos.upm.to_numpy()
    cv = StratifiedGroupKFold(5, shuffle=True, random_state=2026)
    for ent, val in cv.split(datos, y, grupos):
        assert set(grupos[ent]).isdisjoint(grupos[val])
        transformar = ColumnTransformer([
            ('num', SimpleImputer(strategy='median'), numericas),
            ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                                  OneHotEncoder(handle_unknown='ignore')), categoricas)])
        bosque = RandomForestClassifier(n_estimators=250, min_samples_leaf=5, random_state=2026, n_jobs=2)
        modelo = make_pipeline(transformar, bosque)
        modelo.fit(datos.iloc[ent][columnas], y[ent],
                   randomforestclassifier__sample_weight=w[ent]/w[ent].mean())
        pred[val] = modelo.predict(datos.iloc[val][columnas])
        base[val] = np.bincount(y[ent], weights=w[ent]).argmax()
    assert (pred >= 0).all()
    for nombre, p in [('Random Forest', pred), ('Clase mayoritaria', base)]:
        print(f'{nombre}: exactitud ponderada {accuracy_score(y,p,sample_weight=w):.3f}, '
              f'F1 macro con pesos de encuesta {f1_score(y,p,labels=clases,average="macro",sample_weight=w,zero_division=0):.3f}')
    detalle = classification_report(y, pred, labels=clases, sample_weight=w,
                                   output_dict=True, zero_division=0)
    print('Nivel | n observado | sensibilidad ponderada')
    for c in clases:
        print(f'{etiquetas[c]} | {(y==c).sum()} | {detalle[str(c)]["recall"]:.3f}')

In [ ]:
rutas = [base / "outputs/private/analysis.parquet" for base in
         (Path.cwd(), Path.cwd().parent, Path("Proyecto_EH2025_ML_Caso1"))]
ruta = next((p for p in rutas if p.is_file()), None)
if ruta is None:
    print("Para repetir el ajuste, ejecuta esta sección en la carpeta privada del proyecto que contiene analysis.parquet.")
else:
    evaluar_nivel(ruta)